In [ ]:
!pip install ctgan

from ctgan import CTGAN
import pandas as pd

# Load your data
df = pd.read_csv("Heart Failure.csv")

# List your categorical columns exactly as in your CSV
discrete_columns = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# Initialize and train CTGAN
ctgan = CTGAN(epochs=300)
ctgan.fit(df, discrete_columns)

# Generate 500 new synthetic records
new_samples = ctgan.sample(500)

# Concatenate and save
df_augmented = pd.concat([df, new_samples], ignore_index=True)
df_augmented.to_csv("Heart Failure-augmented-ctgan.csv", index=False)
print("Saved: Heart Failure-augmented-ctgan.csv")


In [ ]:
#CNN
# Step 0: Install dependencies (run once)
!pip install scikit-learn pandas numpy torch

# Step 1: Imports
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Step 2: Load and preprocess data
df = pd.read_csv("Heart Failure-augmented-ctgan.csv")  # Ensure this file is accessible

num_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
cat_cols = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
label_col = 'HeartDisease'

# OneHot encode categoricals and scale numerics
X = df[num_cols + cat_cols]
y = df[label_col].astype(int).values

enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = enc.fit_transform(X[cat_cols])

scaler = StandardScaler()
X_num = scaler.fit_transform(X[num_cols])

X_all = np.hstack([X_num, X_cat]).astype(np.float32)
input_dim = X_all.shape[1]

# Step 3: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

# Step 4: Define Denoising Autoencoder (DAE) with larger bottleneck
class DAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, bottleneck_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_rec = self.decoder(z)
        return x_rec

    def encode(self, x):
        return self.encoder(x)

dae = DAE(input_dim=input_dim).to(device)
optimizer_ae = torch.optim.Adam(dae.parameters(), lr=0.001)
criterion_ae = nn.MSELoss()

X_train_torch = torch.tensor(X_train, device=device)
epochs_ae = 150
batch_size_ae = 64

dae.train()
for epoch in range(epochs_ae):
    perm = torch.randperm(X_train_torch.size(0))
    losses = []
    for i in range(0, X_train_torch.size(0), batch_size_ae):
        idx = perm[i:i+batch_size_ae]
        batch = X_train_torch[idx]
        noise = 0.05 * torch.randn_like(batch)
        noisy_batch = torch.clamp(batch + noise, -3, 3)
        rec = dae(noisy_batch)
        loss = criterion_ae(rec, batch)
        optimizer_ae.zero_grad()
        loss.backward()
        optimizer_ae.step()
        losses.append(loss.item())
    if epoch % 10 == 0:
        print(f"DAE Epoch {epoch:3d}: loss={np.mean(losses):.5f}")

dae.eval()
with torch.no_grad():
    X_train_enc = dae.encode(torch.tensor(X_train, device=device)).cpu().numpy()
    X_test_enc = dae.encode(torch.tensor(X_test, device=device)).cpu().numpy()

# Step 5: Define Enhanced CNN Classifier
class EnhancedCNNClassifier(nn.Module):
    def __init__(self, input_length):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32, momentum=0.9),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64, momentum=0.9),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128, momentum=0.9),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.pool(x)
        return self.fc(x)

# Prepare Data for CNN
X_train_cnn = torch.tensor(X_train_enc, dtype=torch.float32, device=device).unsqueeze(1)  # (batch, 1, length)
y_train_cnn = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
X_test_cnn = torch.tensor(X_test_enc, dtype=torch.float32, device=device).unsqueeze(1)
y_test_cnn = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32, device=device)

cnn = EnhancedCNNClassifier(input_length=X_train_enc.shape[1]).to(device)

# Use class weights for imbalance handling
pos_weight = torch.tensor(
    float((y_train == 0).sum()) / (y_train == 1).sum()
).to(device)
criterion_cnn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer_cnn = torch.optim.Adam(cnn.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_cnn, mode='min', factor=0.5, patience=10, verbose=True)

epochs_cnn = 200
batch_size_cnn = 64

# Step 6: Train CNN classifier
cnn.train()
for epoch in range(epochs_cnn):
    perm = torch.randperm(X_train_cnn.size(0))
    losses = []
    for i in range(0, X_train_cnn.size(0), batch_size_cnn):
        idx = perm[i:i+batch_size_cnn]
        xb = X_train_cnn[idx]
        yb = y_train_cnn[idx]
        logits = cnn(xb)
        loss = criterion_cnn(logits, yb)
        optimizer_cnn.zero_grad()
        loss.backward()
        optimizer_cnn.step()
        losses.append(loss.item())
    avg_loss = np.mean(losses)
    scheduler.step(avg_loss)
    if epoch % 10 == 0:
        print(f"CNN Epoch {epoch:3d}: loss={avg_loss:.5f}")

# Step 7: Evaluate CNN classifier
cnn.eval()
with torch.no_grad():
    logits_test = cnn(X_test_cnn)
    preds = torch.sigmoid(logits_test).cpu().numpy().flatten()
    y_pred = (preds >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print("\n--- Enhanced CNN Classifier Results ---")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(classification_report(y_test, y_pred))


In [ ]:
#MLP
!pip install scikit-learn pandas numpy --quiet

# Step 1: Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")

# Step 2: Load and preprocess data
df = pd.read_csv("Heart Failure.csv")

num_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
cat_cols = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
label_col = 'HeartDisease'

X_num = df[num_cols]
X_cat = df[cat_cols]
y = df[label_col].astype(int).values

# Polynomial features on numeric columns (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_num_poly = poly.fit_transform(X_num)

# One-hot encode categorical features
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_enc = enc.fit_transform(X_cat)

# Scale numeric polynomial features
scaler = StandardScaler()
X_num_poly_scaled = scaler.fit_transform(X_num_poly)

# Combine numeric and categorical features
X_all = np.hstack([X_num_poly_scaled, X_cat_enc]).astype(np.float32)

# Step 3: Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

# Step 4: Train RandomForest Classifier
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Step 5: Evaluate on test set
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n--- Final Test Results ---")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(classification_report(y_test, y_pred))

# Optional: Feature importance
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
print("\nTop 10 Important Features:")
for i in range(10):
    print(f"{i+1}. Feature {indices[i]} - Importance {importances[indices[i]]:.4f}")


In [ ]:
#VAE+MLP
# Step 0: Install dependencies (run once)
!pip install scikit-learn pandas numpy torch imbalanced-learn --quiet

# Step 1: Imports
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Step 2: Load and preprocess data
df = pd.read_csv("Heart Failure-augmented-ctgan.csv")
num_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
cat_cols = ['Sex', 'ChestPainType', 'FastingBS', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
label_col = 'HeartDisease'
X_num = df[num_cols]
X_cat = df[cat_cols]
y = df[label_col].astype(int).values

poly = PolynomialFeatures(degree=2, include_bias=False)
X_num_poly = poly.fit_transform(X_num)
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_enc = enc.fit_transform(X_cat)
scaler = StandardScaler()
X_num_poly_scaled = scaler.fit_transform(X_num_poly)
X_all = np.hstack([X_num_poly_scaled, X_cat_enc]).astype(np.float32)

# Step 3: Train/test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

# Step 4: SMOTE balance
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_full, y_train_full)
print(f"SMOTE distribution: {np.bincount(y_train_bal)}")

# Step 5: 5-Fold Stratified CV for reliable tuning
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model grid for search
latent_dims = [48, 64, 96]  # try more for heavier compute
mlp_widths = [64, 128]
dropouts = [0.3]

# Early stop and optimizer params
epochs_vae = 100
epochs_mlp = 70
patience = 10

# Step 6: Model Definitions
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, latent_dim=64, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc21 = nn.Linear(hidden_dim, latent_dim)
        self.fc22 = nn.Linear(hidden_dim, latent_dim)
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def encode(self, x):
        h1 = self.relu(self.bn1(self.fc1(x)))
        h1 = self.dropout(h1)
        mu = self.fc21(h1)
        logvar = self.fc22(h1)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z):
        h3 = self.relu(self.bn3(self.fc3(z)))
        h3 = self.dropout(h3)
        return self.fc4(h3)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def get_latent(self, x):
        self.eval()
        with torch.no_grad():
            mu, _ = self.encode(x)
        return mu

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction='mean')
    kl_div = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_div

class MLP(nn.Module):
    def __init__(self, input_dim, h1=128, h2=64, h3=32, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h3, 1)
        )
    def forward(self, x):
        return self.network(x)

# Step 7: Training and Grid Search
results = []
for latent_dim in latent_dims:
    for mlp_width in mlp_widths:
        for dropout in dropouts:
            fold_accuracies = []
            fold_f1s = []
            fold_thresholds = []

            print(f"\nTesting latent_dim={latent_dim}, mlp_width={mlp_width}, dropout={dropout}")
            for train_idx, val_idx in skf.split(X_train_bal, y_train_bal):
                # K-fold split
                X_train, X_val = X_train_bal[train_idx], X_train_bal[val_idx]
                y_train, y_val = y_train_bal[train_idx], y_train_bal[val_idx]
                # ----- VAE -----
                vae = VAE(input_dim=X_all.shape[1], latent_dim=latent_dim, dropout=dropout).to(device)
                optimizer_vae = torch.optim.AdamW(vae.parameters(), lr=0.001, weight_decay=1e-5)
                X_train_torch = torch.tensor(X_train, dtype=torch.float32, device=device)
                vae.train()
                for epoch in range(epochs_vae):
                    perm = torch.randperm(X_train_torch.size(0))
                    losses = []
                    vae.train()
                    for i in range(0, X_train_torch.size(0), 64):
                        idx = perm[i:i+64]
                        batch = X_train_torch[idx]
                        noise = 0.05 * torch.randn_like(batch)
                        noisy_batch = torch.clamp(batch + noise, -3, 3)
                        recon_batch, mu, logvar = vae(noisy_batch)
                        loss = vae_loss(recon_batch, batch, mu, logvar, beta=1.0)
                        optimizer_vae.zero_grad()
                        loss.backward()
                        optimizer_vae.step()
                        losses.append(loss.item())
                    if epoch > 0 and epoch % 20 == 0:
                        print(f"    VAE Epoch {epoch:>3} loss={np.mean(losses):.5f}")
                vae.eval()
                with torch.no_grad():
                    X_train_enc = vae.get_latent(torch.tensor(X_train, device=device)).cpu().numpy()
                    X_val_enc = vae.get_latent(torch.tensor(X_val, device=device)).cpu().numpy()
                # ----- MLP -----
                mlp = MLP(input_dim=latent_dim, h1=mlp_width, h2=int(mlp_width/2), h3=int(mlp_width/4), dropout=dropout).to(device)
                optimizer_mlp = torch.optim.AdamW(mlp.parameters(), lr=0.0005, weight_decay=1e-5)
                criterion_mlp = nn.BCEWithLogitsLoss()
                X_train_mlp = torch.tensor(X_train_enc, dtype=torch.float32, device=device)
                y_train_mlp = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32, device=device)
                X_val_mlp = torch.tensor(X_val_enc, dtype=torch.float32, device=device)
                y_val_mlp = torch.tensor(y_val.reshape(-1,1), dtype=torch.float32, device=device)
                # Early stopping
                best_val_loss = float("inf")
                bad_epochs = 0
                best_weights = None
                mlp.train()
                for epoch in range(epochs_mlp):
                    perm = torch.randperm(X_train_mlp.size(0))
                    losses = []
                    for i in range(0, X_train_mlp.size(0), 32):
                        idx = perm[i:i+32]
                        xb = X_train_mlp[idx]
                        yb = y_train_mlp[idx]
                        logits = mlp(xb)
                        loss = criterion_mlp(logits, yb)
                        optimizer_mlp.zero_grad()
                        loss.backward()
                        optimizer_mlp.step()
                        losses.append(loss.item())
                    mlp.eval()
                    with torch.no_grad():
                        val_logits = mlp(X_val_mlp)
                        val_loss = criterion_mlp(val_logits, y_val_mlp).item()
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        best_weights = mlp.state_dict()
                        bad_epochs = 0
                    else:
                        bad_epochs += 1
                    if bad_epochs > patience:
                        # print(f"      Early stopping MLP at epoch {epoch}")
                        break
                    mlp.train()
                mlp.load_state_dict(best_weights if best_weights is not None else mlp.state_dict())
                mlp.eval()
                # Tune threshold for F1
                with torch.no_grad():
                    val_probs = torch.sigmoid(mlp(X_val_mlp)).cpu().numpy().flatten()
                thresholds = np.arange(0.30, 0.80, 0.02)
                f1s = [f1_score(y_val, (val_probs >= t).astype(int)) for t in thresholds]
                best_thr = thresholds[np.argmax(f1s)]
                fold_thresholds.append(best_thr)
                val_preds = (val_probs >= best_thr).astype(int)
                acc = accuracy_score(y_val, val_preds)
                f1 = f1_score(y_val, val_preds)
                fold_accuracies.append(acc)
                fold_f1s.append(f1)
            avg_acc = np.mean(fold_accuracies)
            avg_f1 = np.mean(fold_f1s)
            best_thr = np.median(fold_thresholds)
            print(f"  => CV Accuracy: {avg_acc:.4f}  F1: {avg_f1:.4f}  (median best threshold={best_thr:.2f})")
            results.append((avg_acc, avg_f1, latent_dim, mlp_width, dropout, best_thr))

# Get best config
results.sort(reverse=True)
best_conf = results[0]
_, _, best_latent_dim, best_mlp_width, best_dropout, best_thr = best_conf
print("\nBEST CONFIG: latent_dim={}, mlp_width={}, dropout={}, threshold={:.2f}".format(
    best_latent_dim, best_mlp_width, best_dropout, best_thr
))

# Step 8: Retrain on all balanced train with best config; test on test set
vae = VAE(input_dim=X_all.shape[1], latent_dim=best_latent_dim, dropout=best_dropout).to(device)
optimizer_vae = torch.optim.AdamW(vae.parameters(), lr=0.001, weight_decay=1e-5)
X_train_torch = torch.tensor(X_train_bal, dtype=torch.float32, device=device)
vae.train()
for epoch in range(epochs_vae):
    perm = torch.randperm(X_train_torch.size(0))
    losses = []
    for i in range(0, X_train_torch.size(0), 64):
        idx = perm[i:i+64]
        batch = X_train_torch[idx]
        noise = 0.05 * torch.randn_like(batch)
        noisy_batch = torch.clamp(batch + noise, -3, 3)
        recon_batch, mu, logvar = vae(noisy_batch)
        loss = vae_loss(recon_batch, batch, mu, logvar)
        optimizer_vae.zero_grad()
        loss.backward()
        optimizer_vae.step()
        losses.append(loss.item())
    if epoch > 0 and epoch % 25 == 0:
        print(f"  Final VAE Epoch {epoch:>3} loss={np.mean(losses):.5f}")
vae.eval()
with torch.no_grad():
    X_train_enc = vae.get_latent(torch.tensor(X_train_bal, device=device)).cpu().numpy()
    X_test_enc = vae.get_latent(torch.tensor(X_test, device=device)).cpu().numpy()

mlp = MLP(input_dim=best_latent_dim, h1=best_mlp_width,
          h2=int(best_mlp_width/2), h3=int(best_mlp_width/4), dropout=best_dropout).to(device)
optimizer_mlp = torch.optim.AdamW(mlp.parameters(), lr=0.0005, weight_decay=1e-5)
criterion_mlp = nn.BCEWithLogitsLoss()
X_train_mlp = torch.tensor(X_train_enc, dtype=torch.float32, device=device)
y_train_mlp = torch.tensor(y_train_bal.reshape(-1,1), dtype=torch.float32, device=device)
X_test_mlp = torch.tensor(X_test_enc, dtype=torch.float32, device=device)
y_test_mlp = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32, device=device)

# Early stopping for final model
best_val_loss = float("inf")
bad_epochs = 0
best_weights = None
mlp.train()
for epoch in range(epochs_mlp):
    perm = torch.randperm(X_train_mlp.size(0))
    losses = []
    for i in range(0, X_train_mlp.size(0), 32):
        idx = perm[i:i+32]
        xb = X_train_mlp[idx]
        yb = y_train_mlp[idx]
        logits = mlp(xb)
        loss = criterion_mlp(logits, yb)
        optimizer_mlp.zero_grad()
        loss.backward()
        optimizer_mlp.step()
        losses.append(loss.item())
    mlp.eval()
    with torch.no_grad():
        val_logits = mlp(X_train_mlp)
        val_loss = criterion_mlp(val_logits, y_train_mlp).item()
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = mlp.state_dict()
        bad_epochs = 0
    else:
        bad_epochs += 1
    if bad_epochs > patience:
        # print(f"  Early stopping final MLP at epoch {epoch}")
        break
    mlp.train()
mlp.load_state_dict(best_weights if best_weights is not None else mlp.state_dict())
mlp.eval()

# Step 9: Evaluate on test set using tuned threshold
with torch.no_grad():
    test_probs = torch.sigmoid(mlp(X_test_mlp)).cpu().numpy().flatten()
    test_preds = (test_probs >= best_thr).astype(int)
acc = accuracy_score(y_test, test_preds)
f1 = f1_score(y_test, test_preds)
print("\n=== FINAL TEST RESULTS ===")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(classification_report(y_test, test_preds))
